In [147]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

In [148]:
base_dir = '/data/aman_singh/acuuracy_check'
channel = 'MT'
last_month = '2026-05-31'
run_month = '2026-06-30'

In [149]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [150]:
def discover_channel(file_path):
    #file_path = file_path.split('\\')[2]

    if 'ecom' in file_path:
        return 'ECOM'
    elif 'qcom' in file_path:
        return 'QCOM'
    elif 'mt' in file_path:
        return 'MT'
    elif 'gt' in file_path:
        return 'GT'
    else:
        return 'Channel not found'

    # return "MT"


In [151]:
discover_channel('/data/aman_singh/acuuracy_check/prophet_data_train_till_28_Feb_2026 (7)_mt.csv')

'MT'

In [152]:
def get_run_month(train_till_str):
    return datetime.strptime(train_till_str, 'train_till_%d_%b_%Y') + MonthEnd(1)

In [153]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.



Credentials retrieved successfully for prod db.


In [154]:
# query = f""" 
# SELECT 
#     channel_name,
#     asm_area_code, 
#     depot_code, 
#     parent_material1_code, 
#     month_date, 
#     sum(sec_actuals_vol_rum_month) Sec_Vol_Actuals_Rum_Month,
#     sum(sec_apo_plan_vol_rum_month) Sec_Vol_Apo_Plan_Rum_Month
# FROM (
#     Select 
#         Month_Date, 
#         Distributor_Code, 
#         material_code, 
#         sec_actuals_vol_rum_month, 
#         sec_apo_plan_vol_rum_month
#     from 
#         dwh_bpm_dist_brand_mth_sbp 
#     where 
#     month_date  between '2023-01-01' and '2026-02-28') A

# JOIN (
#     SELECT
#         channel_name, 
#         customer_code, 
#         asm_area_code, 
#         depot_code
#     FROM 
#         mst_customer 
#     WHERE 
#         company_code= 'MIL' 
#         and latest_record_ind=1 
#         and channel_name in ('MT', 'E-Commerce', 'Q-Commerce', 'GT')) C 
#     ON 
#         distributor_code = customer_code
# JOIN (
#     Select 
#         material_code, 
#         parent_material1_code 
#     from 
#         mst_material 
#     where 
#         company_code= 'MIL' 
#         and latest_record_ind=1) M 
#     ON 
#         A.material_code = M.material_code
#     group by 
#         channel_name, asm_area_code, depot_code, parent_material1_code, Month_date
#     ORDER BY 
#         channel_name, asm_area_code, depot_code, parent_material1_code, Month_date
# """
# # GT
# results = pd.read_sql(con=prod_conn, sql=query)
# sales_data = pd.DataFrame(results)
# sales_data.columns = sales_data.columns.str.lower()
# sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

In [155]:
query = f"""SELECT
    CM.channel_name,
    CM.asm_area_code,
    CM.depot_code,
    MM.parent_material_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(MESR.sec_apo_plan_vol_rum) AS Sec_Vol_Apo_Plan_Rum_Month,
    SUM(MESR.sec_actuals_vol_rum) AS Sec_Vol_Actuals_Rum_Month
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
        
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        channel_name, 
        asm_area_code,
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
        AND channel_name = '{channel}'
) CM ON MESR.distributor_code = CM.customer_code
WHERE
    month_date BETWEEN '2023-01-01' and '{last_month}'
    
GROUP BY 1, 2, 3, 4,5
ORDER BY 1, 2, 3, 4,5
"""

results = pd.read_sql(con=prod_conn, sql=query)
sales_data = pd.DataFrame(results)
sales_data.columns = sales_data.columns.str.lower()
sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})
sales_data

,channel_name,asm_area_code,depot_code,parent_material_code,month_date,sec_vol_apo_plan_rum_month,sec_vol_actuals_rum_month
0,MT,BCE1,D231,702478,2024-09-30,0.0,0.0
1,MT,BCE1,D231,702478,2024-10-31,0.0,0.0
2,MT,BCE1,D231,702478,2024-12-31,0.0,0.0
3,MT,BCE1,D231,705148,2024-09-30,0.0,0.0
4,MT,BCE1,D231,705148,2024-10-31,0.0,0.0
...,...,...,...,...,...,...,...
646084,MT,MTS,D674,809055,2025-03-31,0.0,0.0
646085,MT,MTS,D674,809057,2025-03-31,0.0,0.0
646086,MT,MTS,D674,809238,2025-03-31,0.0,0.0
646087,MT,MTS,D674,809239,2025-03-31,0.0,0.0


In [156]:
# query = f"""SELECT
#     CM.channel_name,
#     CM.asm_area_code,
#     CM.depot_code,
#     MM.parent_material_code,
#     LAST_DAY(MESR.month_date) AS month_date,
#     SUM(MESR.sec_apo_plan_vol_rum) AS Sec_Vol_Apo_Plan_Rum_Month,
#     SUM(MESR.sec_actuals_vol_rum) AS Sec_Vol_Actuals_Rum_Month
# FROM
#     dwh_bpm_dist_sku_daily MESR
# JOIN
# (
#     SELECT
#         material_code,
#         parent_material_code,
#         material_group_code,
#         uom_reporting,
#         vol_per_unit
#     FROM 
#         mst_material
#     WHERE
#         company_code='MIL' AND
#         latest_record_ind=1
        
# ) MM ON MESR.material_code = MM.material_code
# JOIN
# (
#     SELECT DISTINCT
#         channel_name, 
#         asm_area_code,
#         customer_code,
#         depot_code
#     FROM
#         mst_customer
#     WHERE
#         company_code='MIL' AND
#         latest_record_ind=1
#         AND channel_name in ('E-Commerce', 'GT', 'MT', 'Q-Commerce') 
# ) CM ON MESR.distributor_code = CM.customer_code
# WHERE
#     month_date BETWEEN '2023-01-01' and '{last_month}'
    
# GROUP BY 1, 2, 3, 4,5
# ORDER BY 1, 2, 3, 4,5
# """

# results = pd.read_sql(con=prod_conn, sql=query)
# sales_data = pd.DataFrame(results)
# sales_data.columns = sales_data.columns.str.lower()
# sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})
# sales_data

In [157]:
sales_data['channel_name'].unique()

array(['MT'], dtype=object)

In [158]:
# sales_data['key'] = (
#     sales_data['asm_area_code'].astype(str) + '_' +
#     sales_data['depot_code'].astype(str) + '_' +
#     sales_data['parent_material_code'].astype(str)
# )
# sales_data[sales_data['key'] == 'WMP_D464_718458']

In [159]:
# sales_data.groupby(['key'])['sec_vol_actuals_rum_month'].sum().reset_index().sort_values(by = ['sec_vol_actuals_rum_month'])

In [160]:
sales_data.isnull().sum()

channel_name                  0
asm_area_code                 0
depot_code                    0
parent_material_code          0
month_date                    0
sec_vol_apo_plan_rum_month    0
sec_vol_actuals_rum_month     0
dtype: int64

In [161]:
sales_data['month_date'].min()

datetime.date(2023, 1, 31)

In [162]:
realignment_df = pd.read_sql(
    'select * from trn_mil_asm_psku_realignment',
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def realign_pskus(data, channel, columns=['sec_vol_actuals_rum_month', 'pri_actuals_vol_rum_month']):

    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku


    data = data.groupby(
        ["channel_name", "asm_area_code", 'depot_code', "parent_material_code", "month_date"],
        as_index=False,
    )[columns].agg('sum')

    return data

In [163]:
sales_data['channel_name'].replace({'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'}, inplace=True)

In [164]:
sales_data['channel_name'].unique()

array(['MT'], dtype=object)

In [165]:
sales_data['month_date'] = pd.to_datetime(sales_data['month_date'])

In [166]:
realigned_df = pd.DataFrame()

for channel in sales_data['channel_name'].unique():
    tmp_df = sales_data[sales_data['channel_name'] == channel]
    tmp_df = realign_pskus(tmp_df, channel=channel, columns=['sec_vol_actuals_rum_month'])
    realigned_df = pd.concat([realigned_df, tmp_df])
    del tmp_df

In [167]:
realigned_df.duplicated(
    subset=['channel_name', 'asm_area_code', 'depot_code', 
            'parent_material_code', 'month_date']
).sum()

0

In [168]:
realigned_df['key'] = realigned_df['asm_area_code'] + '_' + realigned_df['depot_code'] + '_' +  realigned_df['parent_material_code'].astype(str) 
realigned_df['month_date'] = pd.to_datetime(realigned_df['month_date'])

realigned_df.duplicated(subset=['channel_name', 'key', 'month_date']).sum()

0

In [169]:
realigned_df = realigned_df.groupby(
    ['channel_name', 'key', 'asm_area_code', 'depot_code', 'parent_material_code', 'month_date'],
    as_index=False
)['sec_vol_actuals_rum_month'].sum()

In [170]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [171]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                read_file['channel'] = channel
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [172]:
channel

'MT'

In [173]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')
# data_file_df = collate_file('\\data.csv')

/data/aman_singh/acuuracy_check/trend_file_train_till_31_May_2026 (7).csv
/data/aman_singh/acuuracy_check/trend_file_train_till_31_May_2026 (6).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_May_2026 (6).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_May_2026 (7).csv


In [174]:
trend_file_df.head()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,qtr_ind_rate,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path
0,BCE1_D231_718322,2023-01-31,0.166667,0.16,0.206801,0.054444,0.002814,0.002701,0.003491,0.000919,MT,BCE1,D231,718322,SAFF KO,0.50,0,0,0,0,0,0,168827.536176,0.008441,NaN,NaN,0.50,0.008441,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
1,BCE1_D231_718322,2023-02-28,0.166667,0.16,0.175505,0.052444,0.002814,0.002701,0.002963,0.000885,MT,BCE1,D231,718322,SAFF KO,0.00,0,0,0,0,0,0,168827.536176,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
2,BCE1_D231_718322,2023-03-31,0.166667,0.16,0.125700,0.038889,0.002814,0.002701,0.002122,0.000657,MT,BCE1,D231,718322,SAFF KO,0.00,0,0,0,0,0,0,168827.536176,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
3,BCE1_D231_718322,2023-04-30,0.166667,0.16,0.116227,0.035778,0.002814,0.002701,0.001962,0.000604,MT,BCE1,D231,718322,SAFF KO,0.16,0,0,0,0,0,0,168827.536176,0.002701,NaN,NaN,0.16,0.002701,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
4,BCE1_D231_718322,2023-05-31,0.053333,0.16,0.160977,0.046889,0.000900,0.002701,0.002718,0.000792,MT,BCE1,D231,718322,SAFF KO,0.16,0,0,0,0,0,0,168827.536176,0.002701,NaN,NaN,0.16,0.002701,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...


In [175]:
trend_file_df['channel'].unique()

array(['MT'], dtype=object)

In [176]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month',
       'diwali', 'diwali_lead_1', 'diwali_lead_2', 'ganesh_chaturthi',
       'ganesh_chaturthi_lead_1', 'ganesh_chaturthi_lead_2', 'qtr_ind_rate',
       'sec_vol_actuals_rum_month_value', 'pred_best_model',
       'pred_value_best_model', 'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path'],
      dtype='object')

In [177]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])
# data_file_df['month_date'] = pd.to_datetime(data_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till']) + MonthEnd(1)
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till']) + MonthEnd(1)

assert (trend_file_df['run_month'] == pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))).all()
assert (prophet_file_df['run_month'] == pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))).all()

In [178]:
# data_file_df['run_month'] = pd.to_datetime(data_file_df['run_month'])

In [179]:
# data_file_df.columns

In [180]:
# missing_forecasts = pd.DataFrame()

# for rm_dt in trend_file_df['run_month'].unique():
#     for c in trend_file_df['channel'].unique():
#         tmp_df = data_file_df[
#             (data_file_df['run_month'] == rm_dt) &
#             (data_file_df['channel'] == c) &
#             (~data_file_df['key'].isin(trend_file_df[
#                 (trend_file_df['run_month'] == rm_dt) &
#                 (trend_file_df['channel'] == c)
#             ]['key'].unique()))
#         ]

#         missing_forecasts = pd.concat([missing_forecasts, tmp_df]).reset_index(drop=True)
    

In [181]:
# data_file_df.duplicated(
#     subset=['channel', 'run_month', 'month_date', 'asm_area_code',
#             'depot_code', 'parent_material_code']
# ).sum()

In [182]:
# missing_forecasts['tmp_key'] = missing_forecasts[['channel', 'key', 'run_month']].astype(str).agg('_'.join, axis=1)

In [183]:
# tmp_trend_df = trend_file_df.copy()
# tmp_trend_df['tmp_key'] = trend_file_df[['channel', 'key', 'run_month']].astype(str).agg('_'.join, axis=1)

# set(missing_forecasts['tmp_key'].unique()).intersection(tmp_trend_df['tmp_key'].unique())

In [184]:
# del tmp_trend_df, missing_forecasts['tmp_key']

In [185]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month',
       'diwali', 'diwali_lead_1', 'diwali_lead_2', 'ganesh_chaturthi',
       'ganesh_chaturthi_lead_1', 'ganesh_chaturthi_lead_2', 'qtr_ind_rate',
       'sec_vol_actuals_rum_month_value', 'pred_best_model',
       'pred_value_best_model', 'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'run_month'],
      dtype='object')

In [186]:
trend_file_df[trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month'])]

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,qtr_ind_rate,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,run_month


In [187]:
trend_file_df['channel'].unique() #, prophet_file_df['channel'].unique()

array(['MT'], dtype=object)

In [188]:
for channel in trend_file_df['channel'].unique():
    print(trend_file_df[trend_file_df['channel'] == channel]['asm_area_code'].unique())

['BCE1' 'BCE2' 'BCN1' 'BCN2' 'BCS1' 'BCS2' 'BCW1' 'BCW2' 'MCE1' 'MCE2'
 'MCN1' 'MCN2' 'MCS1' 'MCS2' 'MCW1' 'MCW2']


In [189]:
# trend_file_df.groupby(['key'])['sec_vol_actuals_rum_month'].sum().reset_index().sort_values(by = ['sec_vol_actuals_rum_month'])

In [190]:
trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum(), prophet_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum()

(0, 0)

In [191]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-06-30 00:00:00'): {Timestamp('2026-06-30 00:00:00'): 'M',
  Timestamp('2026-07-31 00:00:00'): 'M+1',
  Timestamp('2026-08-31 00:00:00'): 'M+2',
  Timestamp('2026-09-30 00:00:00'): 'M+3',
  Timestamp('2026-10-31 00:00:00'): 'M+4',
  Timestamp('2026-11-30 00:00:00'): 'M+5',
  Timestamp('2026-12-31 00:00:00'): 'M+6',
  Timestamp('2027-01-31 00:00:00'): 'M+7',
  Timestamp('2027-02-28 00:00:00'): 'M+8'}}

In [192]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [193]:
# missing_forecasts['M month'] = missing_forecasts.apply(
#     lambda x: mappings[x['run_month']].get(
#         x['month_date']
#     ), axis=1
# )

In [194]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-06-30,2026-05-31


In [195]:
# prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

In [196]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4', 'M+5', 'M+6', 'M+7'],
      dtype=object)

In [197]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,brand_code,sec_vol_actuals_rum_month,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,qtr_ind_rate,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,run_month,M month
0,BCE1_D231_718322,2023-01-31,0.166667,0.16,0.206801,0.054444,0.002814,0.002701,0.003491,0.000919,MT,BCE1,D231,718322,SAFF KO,0.50,0,0,0,0,0,0,168827.536176,0.008441,NaN,NaN,0.50,0.008441,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
1,BCE1_D231_718322,2023-02-28,0.166667,0.16,0.175505,0.052444,0.002814,0.002701,0.002963,0.000885,MT,BCE1,D231,718322,SAFF KO,0.00,0,0,0,0,0,0,168827.536176,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
2,BCE1_D231_718322,2023-03-31,0.166667,0.16,0.125700,0.038889,0.002814,0.002701,0.002122,0.000657,MT,BCE1,D231,718322,SAFF KO,0.00,0,0,0,0,0,0,168827.536176,0.000000,NaN,NaN,0.00,0.000000,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
3,BCE1_D231_718322,2023-04-30,0.166667,0.16,0.116227,0.035778,0.002814,0.002701,0.001962,0.000604,MT,BCE1,D231,718322,SAFF KO,0.16,0,0,0,0,0,0,168827.536176,0.002701,NaN,NaN,0.16,0.002701,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
4,BCE1_D231_718322,2023-05-31,0.053333,0.16,0.160977,0.046889,0.000900,0.002701,0.002718,0.000792,MT,BCE1,D231,718322,SAFF KO,0.16,0,0,0,0,0,0,168827.536176,0.002701,NaN,NaN,0.16,0.002701,2026-05-31,1.041699,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109693,MCW2_D463_726283,2026-09-30,0.000000,0.00,0.000000,0.101520,0.000000,0.000000,0.000000,0.003546,MT,MCW2,D463,726283,PCNO(R),0.00,0,0,1,1,0,0,349274.001420,0.000000,0.000000,0.000000,0.00,0.000000,2026-05-31,1.435550,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3
109694,MCW2_D463_726283,2026-10-31,0.000000,0.00,0.008037,0.081000,0.000000,0.000000,0.000281,0.002829,MT,MCW2,D463,726283,PCNO(R),0.00,0,1,0,0,0,0,349274.001420,0.000000,0.008037,0.000281,0.00,0.000000,2026-05-31,1.435550,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4
109695,MCW2_D463_726283,2026-11-30,0.000000,0.00,0.000000,0.006840,0.000000,0.000000,0.000000,0.000239,MT,MCW2,D463,726283,PCNO(R),0.00,1,0,0,0,0,0,349274.001420,0.000000,0.000000,0.000000,0.00,0.000000,2026-05-31,1.435550,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+5
109696,MCW2_D463_726283,2026-12-31,0.000000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MT,MCW2,D463,726283,PCNO(R),0.00,0,0,0,0,0,0,349274.001420,0.000000,0.000000,0.000000,0.00,0.000000,2026-05-31,1.435550,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+6


In [198]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month',
       'diwali', 'diwali_lead_1', 'diwali_lead_2', 'ganesh_chaturthi',
       'ganesh_chaturthi_lead_1', 'ganesh_chaturthi_lead_2', 'qtr_ind_rate',
       'sec_vol_actuals_rum_month_value', 'pred_best_model',
       'pred_value_best_model', 'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'run_month', 'M month'],
      dtype='object')

In [199]:
prophet_file_df[
    ['month_date', 'key', 'channel', 'run_month']
].duplicated().sum()

0

In [200]:
prophet_file_df.columns

Index(['ds', 'trend', 'yhat_lower', 'yhat_upper', 'trend_lower', 'trend_upper',
       'yhat_60_%ile', 'yhat_70_%ile', 'yhat_75_%ile', 'trend_60_%ile',
       'trend_70_%ile', 'trend_75_%ile', 'additive_terms',
       'additive_terms_lower', 'additive_terms_upper', 'diwali',
       'diwali_lower', 'diwali_upper', 'diwali_lead_1', 'diwali_lead_1_lower',
       'diwali_lead_1_upper', 'diwali_lead_2', 'diwali_lead_2_lower',
       'diwali_lead_2_upper', 'extra_regressors_additive',
       'extra_regressors_additive_lower', 'extra_regressors_additive_upper',
       'ganesh_chaturthi', 'ganesh_chaturthi_lower', 'ganesh_chaturthi_upper',
       'ganesh_chaturthi_lead_1', 'ganesh_chaturthi_lead_1_lower',
       'ganesh_chaturthi_lead_1_upper', 'ganesh_chaturthi_lead_2',
       'ganesh_chaturthi_lead_2_lower', 'ganesh_chaturthi_lead_2_upper',
       'yearly', 'yearly_lower', 'yearly_upper', 'multiplicative_terms',
       'multiplicative_terms_lower', 'multiplicative_terms_upper', 'yhat',
     

In [201]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'channel', 'key', 'run_month', 'yhat_60_%ile', 'yhat_70_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'channel', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge
del len_before_merge

In [202]:
# trend_file_df['Skipped'] = 0

In [203]:
# missing_forecasts['Skipped'] = 1

In [204]:
# trend_file_df = pd.concat([trend_file_df, missing_forecasts]).reset_index(drop=True)

In [205]:
trend_file_df.duplicated(subset=['key', 'channel', 'month_date', 'run_month']).sum()

0

In [206]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [207]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [208]:
qtr_ind_df = read_qtr_ind_rate_table()
qtr_ind_df.head()


Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [209]:
trend_file_df['portfolio'].isna().sum()

0

In [210]:
assert qtr_ind_df.duplicated(subset=['brand_code']).sum() == 0
qtr_ind_df.drop('month_date', axis=1, inplace=True)

In [211]:
trend_file_df.drop('qtr_ind_rate', axis=1, inplace=True)

In [212]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    qtr_ind_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [213]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'channel', 'key']
).sum() == 0

In [214]:
trend_file_df.drop('sec_vol_actuals_rum_month', axis=1, inplace=True)

In [215]:
realigned_df.dtypes

channel_name                         object
key                                  object
asm_area_code                        object
depot_code                           object
parent_material_code                  int64
month_date                   datetime64[ns]
sec_vol_actuals_rum_month           float64
dtype: object

In [216]:
assert realigned_df.duplicated(subset=['channel_name', 'key', 'month_date']).sum() == 0

In [217]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    realigned_df[['channel_name', 'key', 'month_date', 'sec_vol_actuals_rum_month']].rename(
        columns={
            'channel_name': 'channel'
        }
    ),
    on=['month_date', 'channel', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [218]:
trend_file_df.select_dtypes('number').isna().sum()

pred_p3m                                        0
pred_p6m                                        0
pred_prophet                                    0
pred_rf                                         0
pred_value_p3m                                  0
pred_value_p6m                                  0
pred_value_prophet                              0
pred_value_rf                                   0
parent_material_code                            0
diwali                                          0
diwali_lead_1                                   0
diwali_lead_2                                   0
ganesh_chaturthi                                0
ganesh_chaturthi_lead_1                         0
ganesh_chaturthi_lead_2                         0
sec_vol_actuals_rum_month_value                 0
pred_best_model                            170728
pred_value_best_model                      170728
sec_vol_actuals_rum_month_treated               0
sec_vol_actuals_rum_month_value_treated         0


In [219]:
trend_file_df = trend_file_df.fillna(0)

In [220]:
trend_file_df.select_dtypes('number').min().round()

pred_p3m                                        0.0
pred_p6m                                        0.0
pred_prophet                                    0.0
pred_rf                                         0.0
pred_value_p3m                                  0.0
pred_value_p6m                                  0.0
pred_value_prophet                              0.0
pred_value_rf                                   0.0
parent_material_code                       718287.0
diwali                                          0.0
diwali_lead_1                                   0.0
diwali_lead_2                                   0.0
ganesh_chaturthi                                0.0
ganesh_chaturthi_lead_1                         0.0
ganesh_chaturthi_lead_2                         0.0
sec_vol_actuals_rum_month_value                 0.0
pred_best_model                                 0.0
pred_value_best_model                           0.0
sec_vol_actuals_rum_month_treated               0.0
sec_vol_actu

In [221]:
# 'pred_prophet_70%ile',
for col in [ 'pred_best_model', 'pred_value_best_model',  'sec_vol_actuals_rum_month', 'pred_prophet_70%ile', 'pred_prophet_60%ile']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [222]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'channel', 'key', 'month_date']
)

In [223]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['sec_vol_actuals_rum_month'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [224]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [225]:
# trend_file_df['P3M'].sum()
trend_file_df[trend_file_df['month_date'] == '2026-02-28']['P3M'].sum()

638196.0296666578

In [226]:
# pd.Series([100, 2, 4, 6]).nlargest(2).mean()

In [227]:
np.sort(np.array([100, 2, 4, 6]))[-2:].mean()

53.0

In [228]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'diwali', 'diwali_lead_1',
       'diwali_lead_2', 'ganesh_chaturthi', 'ganesh_chaturthi_lead_1',
       'ganesh_chaturthi_lead_2', 'sec_vol_actuals_rum_month_value',
       'pred_best_model', 'pred_value_best_model',
       'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'run_month', 'M month', 'pred_prophet_60%ile',
       'pred_prophet_70%ile', 'portfolio', 'qtr_ind_rate',
       'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy'],
      dtype='object')

In [229]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [230]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [231]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'channel', 'key', 'month_date']
)

In [232]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'channel', 'key'])['P3M']
      .pct_change() * 100
)

In [233]:
trend_file_df = trend_file_df.sort_values(
    ['channel', 'run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [234]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'channel', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [235]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [236]:
trend_file_df['month_date'].unique()

<DatetimeArray>
['2023-01-31 00:00:00', '2023-02-28 00:00:00', '2023-03-31 00:00:00',
 '2023-04-30 00:00:00', '2023-05-31 00:00:00', '2023-06-30 00:00:00',
 '2023-07-31 00:00:00', '2023-08-31 00:00:00', '2023-09-30 00:00:00',
 '2023-10-31 00:00:00', '2023-11-30 00:00:00', '2023-12-31 00:00:00',
 '2024-01-31 00:00:00', '2024-02-29 00:00:00', '2024-03-31 00:00:00',
 '2024-04-30 00:00:00', '2024-05-31 00:00:00', '2024-06-30 00:00:00',
 '2024-07-31 00:00:00', '2024-08-31 00:00:00', '2024-09-30 00:00:00',
 '2024-10-31 00:00:00', '2024-11-30 00:00:00', '2024-12-31 00:00:00',
 '2025-01-31 00:00:00', '2025-02-28 00:00:00', '2025-03-31 00:00:00',
 '2025-04-30 00:00:00', '2025-05-31 00:00:00', '2025-06-30 00:00:00',
 '2025-07-31 00:00:00', '2025-08-31 00:00:00', '2025-09-30 00:00:00',
 '2025-10-31 00:00:00', '2025-11-30 00:00:00', '2025-12-31 00:00:00',
 '2026-01-31 00:00:00', '2026-02-28 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00', '2026-05-31 00:00:00', '2026-06-30 00:00:00',
 '20

In [237]:
pd.set_option('display.max_columns', None)

In [238]:
#trend_file_df['LY P3M'].sum()
trend_file_df[trend_file_df['month_date'] == '2026-02-28']['P3M'].sum()

638196.0296666578

In [239]:
trend_file_df[
    (trend_file_df['month_date'] == '2026-02-28') &
    (trend_file_df['P3M'].isna())
]

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,brand_code,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,sec_vol_actuals_rum_month,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)"


In [240]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['key','channel'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [241]:
trend_file_df[trend_file_df['month_date'] == '2026-02-28']['P3M'].sum()

638196.0296666578

In [242]:
trend_file_df['LY P3M'].sum()

21859485.96833306

In [243]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [244]:
trend_file_df['sec_vol_actuals_rum_month_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['sec_vol_actuals_rum_month']/ (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile']/ (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile']/ (10 ** 7)

In [245]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'sec_vol_actuals_rum_month_value',
 'pred_value_best_model',
 'sec_vol_actuals_rum_month_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [246]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

In [247]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,brand_code,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,sec_vol_actuals_rum_month,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
109654,BCS1_D530_718472,2023-01-31,57.6,99.9,263.555732,23.580,0.002862,0.004963,0.013094,0.001172,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.004024,0.0,0.0,81.0,0.004024,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,304.047617,362.682855,Hair Oils,496.828458,81.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.018019,0.015106
109655,BCS1_D530_718472,2023-02-28,57.6,99.9,262.888668,50.700,0.002862,0.004963,0.013061,0.002519,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.003219,0.0,0.0,64.8,0.003219,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,314.587823,360.384120,Hair Oils,496.828458,64.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017905,0.015630
109656,BCS1_D530_718472,2023-03-31,57.6,99.9,242.359795,29.940,0.002862,0.004963,0.012041,0.001488,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.001341,0.0,0.0,27.0,0.001341,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,295.802504,350.144730,Hair Oils,496.828458,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017396,0.014696
109657,BCS1_D530_718472,2023-04-30,57.6,99.9,206.727192,26.220,0.002862,0.004963,0.010271,0.001303,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.001610,0.0,0.0,32.4,0.001610,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,255.622264,313.025255,Hair Oils,496.828458,32.4,57.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.6,0.002862,NaN,NaN,NaN,0.015552,0.012700
109658,BCS1_D530_718472,2023-05-31,41.4,99.9,162.234479,26.100,0.002057,0.004963,0.008060,0.001297,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.007244,0.0,0.0,145.8,0.007244,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,222.720523,277.680580,Hair Oils,496.828458,145.8,41.4,NaN,NaN,NaN,NaN,NaN,NaN,-28.125,NaN,NaN,NaN,41.4,0.002057,NaN,NaN,NaN,0.013796,0.011065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93624,MCW2_D356_809423,2026-09-30,0.0,0.0,0.000000,0.432,0.000000,0.000000,0.000000,0.000074,MT,MCW2,D356,809423,SW_SGPRF,0,0,1,1,0,0,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.647894,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,0.000000,0.000000,Male Grooming,1712.605337,0.0,0.0,0.0,3.6,3.0,2.4,0.0,0.0,-100.000,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.000617,0.000514,0.000000,0.000000
93625,MCW2_D356_809423,2026-10-31,0.0,0.0,0.000000,0.216,0.000000,0.000000,0.000000,0.000037,MT,MCW2,D356,809423,SW_SGPRF,0,1,0,0,0,0,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.647894,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+4,0.000000,0.000000,Male Grooming,1712.605337,0.0,0.0,0.0,3.6,2.4,1.2,0.0,0.0,-100.000,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.000617,0.000411,0.000000,0.000000
93626,MCW2_D356_809423,2026-11-30,0.0,0.0,0.000000,0.576,0

In [248]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'channel', 'month_date']
).sum() == 0

In [249]:
trend_file_df = trend_file_df.sort_values(
    ['channel', 'run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['channel', 'run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(24)


trend_file_df['Sec_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(1)

trend_file_df['Sec_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(2)

trend_file_df['Sec_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['channel','run_month', 'brand_code', 'key']
)['sec_vol_actuals_rum_month_value'].shift(3)

In [250]:
for col in ['Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2', 'Sec_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['channel','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [251]:
trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [252]:
trend_file_df.reset_index(drop=True, inplace=True)

In [253]:
trend_file_df.shape

(208936, 66)

In [254]:
trend_file_df['key'].nunique()

4776

In [255]:
brand_class = pd.read_excel('/data/aman_singh/acuuracy_check/brand_class_new.xlsx')
brand_class['Channel'] = brand_class['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
brand_class.columns = brand_class.columns.str.lower()
brand_class = brand_class[['channel', 'brand','final class']]
brand_class.rename(columns = {'brand':'brand_code','final class':'class'}, inplace = True)


In [256]:
# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()

In [257]:
# brand_class_df.columns = ['brand_code', 'class']

In [258]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_class, 
    on=['channel','brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)
del len_before_merge

In [259]:
# trend_file_df['class'].isna().sum()

In [260]:
trend_file_df['brand_code'].unique()

array(['ADV-AHO-R', 'BIO OILS', 'BRD_BDSPR', 'CO_SO_VCN', 'H&C',
       'H&C DFOIL', 'H&C_ALMND', 'HC SNS', 'KAYA_GM', 'KAYA_ML', 'KERALA',
       'LIVON S-R', 'LVNPST_ML', 'LVN_PRFSR', 'LVN_SRSNS', 'LVN_SR_DR',
       'MALO-NATU', 'MALT-NATU', 'NHR-SABDM', 'NHR-UTTAM', 'NHR_AMLGD',
       'NHR_SSAHO', 'NIHAR NHO', 'NIHAR(R)', 'PA-ALO-HO', 'PA-BDYLOT',
       'PABABY_CM', 'PABABY_GM', 'PABABY_ML', 'PABABY_SP', 'PADV-HOT',
       'PADV-HRCR', 'PADVJAS-R', 'PADV_SMPN', 'PA_CN_HO', 'PA_EXT_ML',
       'PA_GD_PLS', 'PA_HR_MSK', 'PA_JASGLD', 'PA_ONI_HO', 'PCNO FLEX',
       'PCNO(R)', 'P_AL_GOLD', 'P_EN_ALM', 'P_EN_ARG', 'P_EN_BGHB',
       'P_EN_CRSH', 'P_EN_RSMR', 'REV.LQDST', 'REV.ST.', 'REV_LQFRG',
       'SAF-MUSLI', 'SAFF ACTV', 'SAFF GOLD', 'SAFF KO', 'SAFF KOCO',
       'SAFF OATS', 'SAFF SALT', 'SAFF_ODLS', 'SAF_HONEY', 'SAF_MAYO',
       'SAF_MILET', 'SAF_PNBTR', 'SFOAT-CUP', 'SFOATS-FL', 'SFOATS_GD',
       'SF_IM_CHY', 'SF_MNCHPS', 'SF_SOYACN', 'SW HRGEL', 'SW HSPRY',
       'SW

In [261]:
trend_file_df[trend_file_df['class'].isna()]['brand_code'].unique()

array(['PA_GD_PLS'], dtype=object)

In [262]:
# trend_file_df['final class'].unique()

In [263]:
trend_file_df[
    # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
    # (trend_file_df['month_date'] > '2024-06-30') &
    (trend_file_df['M month'].notna()) 
    # (trend_file_df['class'].isin(['B', 'C']))
].shape# .to_csv('Heuristic_GT_ECOM_B&C_v1.csv', index=False)

(208936, 67)

In [264]:
(324496, 61)

(324496, 61)

In [265]:
# trend_file_df[
#     # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (trend_file_df['month_date'] > '2024-06-30') &
#     (trend_file_df['M month'].notna())
#     # (trend_file_df['class'].isin(['B', 'C']))
# ]['Skipped'].unique()

In [266]:
# trend_file_df[
#     # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (trend_file_df['month_date'] > '2024-06-30') &
#     (trend_file_df['M month'].notna())
#     # (trend_file_df['class'].isin(['B', 'C']))
# ].to_csv("ALL Channels Live Run Heuristics Dec'25.csv", index=False)

In [268]:
trend_file_df#['key'].nunique()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,brand_code,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,sec_vol_actuals_rum_month,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,ASM,Depot,PSKU,class
0,BCS1_D530_718472,2023-01-31,57.6,99.9,263.555732,23.580,0.002862,0.004963,0.013094,0.001172,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.004024,0.0,0.0,81.0,0.004024,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,304.047617,362.682855,Hair Oils,496.828458,81.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.018019,0.015106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BCS1,D530,718472,C
1,BCS1_D530_718472,2023-02-28,57.6,99.9,262.888668,50.700,0.002862,0.004963,0.013061,0.002519,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.003219,0.0,0.0,64.8,0.003219,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,314.587823,360.384120,Hair Oils,496.828458,64.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017905,0.015630,NaN,NaN,NaN,NaN,0.004024,NaN,NaN,BCS1,D530,718472,C
2,BCS1_D530_718472,2023-03-31,57.6,99.9,242.359795,29.940,0.002862,0.004963,0.012041,0.001488,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.001341,0.0,0.0,27.0,0.001341,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,295.802504,350.144730,Hair Oils,496.828458,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017396,0.014696,NaN,NaN,NaN,NaN,0.003219,0.004024,NaN,BCS1,D530,718472,C
3,BCS1_D530_718472,2023-04-30,57.6,99.9,206.727192,26.220,0.002862,0.004963,0.010271,0.001303,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.001610,0.0,0.0,32.4,0.001610,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,255.622264,313.025255,Hair Oils,496.828458,32.4,57.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.6,0.002862,NaN,NaN,NaN,0.015552,0.012700,NaN,NaN,NaN,NaN,0.001341,0.003219,0.004024,BCS1,D530,718472,C
4,BCS1_D530_718472,2023-05-31,41.4,99.9,162.234479,26.100,0.002057,0.004963,0.008060,0.001297,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.007244,0.0,0.0,145.8,0.007244,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,222.720523,277.680580,Hair Oils,496.828458,145.8,41.4,NaN,NaN,NaN,NaN,NaN,NaN,-28.125,NaN,NaN,NaN,41.4,0.002057,NaN,NaN,NaN,0.013796,0.011065,NaN,NaN,NaN,NaN,0.001610,0.001341,0.003219,BCS1,D530,718472,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208931,MCW2_D356_809423,2026-09-30,0.0,0.0,0.000000,0.432,0.000000,0.000000,0.000000,0.000074,MT,MCW2,D356,809423,SW_SGPRF,0,0,1,1,0,0,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.647894,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,0.000000,0.000000,Male Grooming,1712.605337,0.0,0.0,0.0,3.6,3.0,2.4,0.0,0.0,-100.000,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.000617,0.000514,0.000000,0.000000,0.0,3.6,0.0,0.000617,0.000000,0.000000,0.000000,MCW2,D356,809423,NPD


In [279]:
trend_file_df.to_csv("/data/aman_singh/acuuracy_check/mt_channels Live Run jun'26.csv", index=False)

### some checks

In [269]:
query = """select * from TRN_MIL_DF_DATA
where run_month = '2026-06-30' """

data = pd.read_sql(con=dev_conn, sql=query)
data.columns = data.columns.str.lower()
data

,month_date,asm_area_code,depot_code,parent_material_code,sec_vol_actuals_rum_month,npd_flag,brand_code,qtr_ind_rate,free,extra_vol_co,other_co,price_off_co,seasonal_month_flag,fill_rate,republic_day,holi,good_friday,maharashtra_day,bakri_eid,independence_day,ganesh_chaturthi,gandhi_jayanthi,dussehra,diwali,christmas,ramzan_eid,new_year,gudi_padwa,channel,run_month,rm_price,raw_material,mrp,ec,pc,btl_flag,grp_npt,npt_aft,npt_morn,grp_pt,pt_eve,npt_eve,print_spends_in_lacs,tv_spends_in_lacs,radio_spends_in_lacs,npt,pt,npd,update_timestamp,drive,outlier
0,2022-04-30,BCE1,D231,718303,0.028,0.0,REV.LQDST,247926.6089,0,0.0,0.0,0.0,0,1.0000,None,0,1,None,0,None,0,None,0,0,None,0,None,None,MT,2026-06-30,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
1,2022-05-31,BCE1,D231,718303,0.028,0.0,REV.LQDST,247926.6089,0,0.0,0.0,0.0,0,0.6667,None,0,0,None,0,None,0,None,0,0,None,1,None,None,MT,2026-06-30,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
2,2022-06-30,BCE1,D231,718303,0.028,0.0,REV.LQDST,247926.6089,0,0.0,0.0,0.0,0,1.0000,None,0,0,None,0,None,0,None,0,0,None,0,None,None,MT,2026-06-30,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
3,2022-07-31,BCE1,D231,718303,0.028,0.0,REV.LQDST,247926.6089,0,0.0,0.0,0.0,0,1.0000,None,0,0,None,1,None,0,None,0,0,None,0,None,None,MT,2026-06-30,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
4,2022-08-31,BCE1,D231,718303,0.042,0.0,REV.LQDST,247926.6089,0,0.0,0.0,0.0,0,1.0000,None,0,0,None,0,None,1,None,0,0,None,0,None,None,MT,2026-06-30,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805790,2024-10-31,VIJN,D572,809046,0.400,1.0,PABABY_GM,366.4850,0,0.0,0.0,0.0,0,NaN,None,0,0,None,0,None,0,None,1,1,None,0,None,None,GT,2026-06-30,None,None,150.0,2.0,2.0,0.0,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0.0,0.0
805791,2024-11-30,VIJN,D572,809046,0.000,1.0,PABABY_GM,366.4850,0,0.0,0.0,0.0,0,NaN,None,0,0,None,0,None,0,None,0,0,None,0,None,None,GT,2026-06-30,None,None,150.0,0.0,0.0,0.0,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0.0,0.0
805792,2024-12-31,VIJN,D572,809046,0.000,1.0,PABABY_GM,366.4850,0,0.0,0.0,0.0,0,NaN,None,0,0,None,0,None,0,None,0,0,None,0,None,None,GT,2026-06-30,None,None,150.0,0.0,0.0,0.0,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0.0,0.0
805793,2025-01-31,VIJN,D572,809046,0.000,1.0,PABABY_GM,366.4850,0,0.0,0.0,0.0,0,NaN,None,0,0,None,0,None,0,None,0,0,None,0,None,None,GT,2026-06-30,None,None,150.0,1.0,1.0,0.0,None,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,0.0,0.0


In [270]:
data['key'] = (
    data['asm_area_code'].astype(str) + '_' +
    data['depot_code'].astype(str) + '_' +
    data['parent_material_code'].astype(str)
)
data = data[data['key'].isin(trend_file_df['key'].unique())]

In [271]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,brand_code,diwali,diwali_lead_1,diwali_lead_2,ganesh_chaturthi,ganesh_chaturthi_lead_1,ganesh_chaturthi_lead_2,sec_vol_actuals_rum_month_value,pred_best_model,pred_value_best_model,sec_vol_actuals_rum_month_treated,sec_vol_actuals_rum_month_value_treated,train_till,cov,run,step,file_path,run_month,M month,pred_prophet_60%ile,pred_prophet_70%ile,portfolio,qtr_ind_rate,sec_vol_actuals_rum_month,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,Sec_Value_in_Cr_lag_1,Sec_Value_in_Cr_lag_2,Sec_Value_in_Cr_lag_3,ASM,Depot,PSKU,class
0,BCS1_D530_718472,2023-01-31,57.6,99.9,263.555732,23.580,0.002862,0.004963,0.013094,0.001172,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.004024,0.0,0.0,81.0,0.004024,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,304.047617,362.682855,Hair Oils,496.828458,81.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.018019,0.015106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BCS1,D530,718472,C
1,BCS1_D530_718472,2023-02-28,57.6,99.9,262.888668,50.700,0.002862,0.004963,0.013061,0.002519,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.003219,0.0,0.0,64.8,0.003219,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,314.587823,360.384120,Hair Oils,496.828458,64.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017905,0.015630,NaN,NaN,NaN,NaN,0.004024,NaN,NaN,BCS1,D530,718472,C
2,BCS1_D530_718472,2023-03-31,57.6,99.9,242.359795,29.940,0.002862,0.004963,0.012041,0.001488,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.001341,0.0,0.0,27.0,0.001341,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,295.802504,350.144730,Hair Oils,496.828458,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017396,0.014696,NaN,NaN,NaN,NaN,0.003219,0.004024,NaN,BCS1,D530,718472,C
3,BCS1_D530_718472,2023-04-30,57.6,99.9,206.727192,26.220,0.002862,0.004963,0.010271,0.001303,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.001610,0.0,0.0,32.4,0.001610,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,255.622264,313.025255,Hair Oils,496.828458,32.4,57.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.6,0.002862,NaN,NaN,NaN,0.015552,0.012700,NaN,NaN,NaN,NaN,0.001341,0.003219,0.004024,BCS1,D530,718472,C
4,BCS1_D530_718472,2023-05-31,41.4,99.9,162.234479,26.100,0.002057,0.004963,0.008060,0.001297,MT,BCS1,D530,718472,ADV-AHO-R,0,0,0,0,0,0,0.007244,0.0,0.0,145.8,0.007244,2026-05-31,2.203326,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,0,222.720523,277.680580,Hair Oils,496.828458,145.8,41.4,NaN,NaN,NaN,NaN,NaN,NaN,-28.125,NaN,NaN,NaN,41.4,0.002057,NaN,NaN,NaN,0.013796,0.011065,NaN,NaN,NaN,NaN,0.001610,0.001341,0.003219,BCS1,D530,718472,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208931,MCW2_D356_809423,2026-09-30,0.0,0.0,0.000000,0.432,0.000000,0.000000,0.000000,0.000074,MT,MCW2,D356,809423,SW_SGPRF,0,0,1,1,0,0,0.000000,0.0,0.0,0.0,0.000000,2026-05-31,1.647894,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-06-30,M+3,0.000000,0.000000,Male Grooming,1712.605337,0.0,0.0,0.0,3.6,3.0,2.4,0.0,0.0,-100.000,-100.0,-100.0,0.0,0.0,0.000000,0.0,0.000617,0.000514,0.000000,0.000000,0.0,3.6,0.0,0.000617,0.000000,0.000000,0.000000,MCW2,D356,809423,NPD


In [272]:
import pandas as pd

as_of_date = pd.to_datetime("2026-06-30")  # month-end for Feb 2026
data['month_date'] = pd.to_datetime(data['month_date'])
filtered = data[
    (data['channel'] == 'MT') &
    (data['month_date'] < as_of_date) &
    (data['month_date'] >= as_of_date - pd.DateOffset(months=3))
]


In [273]:
# assert p3m equals
x = filtered.groupby(['month_date'])['sec_vol_actuals_rum_month'].sum().reset_index()['sec_vol_actuals_rum_month'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-06-30']['P3M'].sum()
assert(int(x)==int(y))

In [274]:
(x,y)

(541553.0906666666, 541553.090666659)

In [275]:
ly_end = as_of_date - pd.DateOffset(years=1)
ly_start = ly_end - pd.DateOffset(months=3)

filtered = data[
    (data['channel'] == 'MT') &
    (data['month_date'] < ly_end) &
    (data['month_date'] >= ly_start )
]


In [276]:
# p3m ly check may not equal but should be close
x = filtered.groupby(['month_date'])['sec_vol_actuals_rum_month'].sum().reset_index()['sec_vol_actuals_rum_month'].mean()
y = trend_file_df[trend_file_df['month_date'] == '2026-06-30']['LY P3M'].sum()
(x,y)

(664408.8383333334, 660038.7049999912)

In [277]:
# p3m consistency check
as_of_date = pd.to_datetime('2026-06-30')

next_3_months = pd.date_range(
    start=as_of_date + pd.offsets.MonthEnd(1),
    periods=3,
    freq='M'
)
for dt in next_3_months:
    p3m_sum = trend_file_df.loc[
        trend_file_df['month_date'] == dt, 'P3M'
    ].sum()
    
    print(f"P3M sum for {dt.date()}: {p3m_sum}")


P3M sum for 2026-07-31: 541553.090666659
P3M sum for 2026-08-31: 541553.090666659
P3M sum for 2026-09-30: 541553.090666659


In [278]:
trend_file_df[trend_file_df['month_date'] == '2026-06-30']['P3M_value'].sum()


116.09170863097262

## extra

In [ ]:
pd.read_sql(con=prod_conn, sql=query)

In [6]:
start_date = '2023-01-31'
end_date = '2026-05-31'
channel = 'GT'

In [7]:
last_date = pd.to_datetime(end_date) - MonthEnd(1)
query = """
        SELECT
            CM.asm_area_code,
            CM.depot_code,
            MM.parent_material_code,
            LAST_DAY(MESR.month_date) AS month_date,
            SUM(MESR.sec_apo_plan_vol_rum) AS Sec_Vol_Apo_Plan_Rum_Month,
            SUM(MESR.sec_actuals_vol_rum) AS Sec_Vol_Actuals_Rum_Month
        FROM
            dwh_bpm_dist_sku_daily MESR
        JOIN
        (
            SELECT
                material_code,
                parent_material_code,
                material_group_code,
                uom_reporting,
                vol_per_unit
            FROM 
                mst_material
            WHERE
                company_code='MIL' AND
                latest_record_ind=1
                
        ) MM ON MESR.material_code = MM.material_code
        JOIN
        (
            SELECT DISTINCT
                channel_name, 
                asm_area_code,
                customer_code,
                depot_code
            FROM
                mst_customer
            WHERE
                company_code='MIL' AND
                latest_record_ind=1
                AND channel_name = '{}'
        ) CM ON MESR.distributor_code = CM.customer_code
        WHERE
            month_date BETWEEN '{}' and '{}'
            
        GROUP BY 1, 2, 3, 4
        ORDER BY 1, 2, 3, 4
        """.format(
        channel, start_date, last_date
    )


results = pd.read_sql(con=prod_conn, sql=query)
results = pd.DataFrame(results)
results

,ASM_AREA_CODE,DEPOT_CODE,PARENT_MATERIAL_CODE,MONTH_DATE,SEC_VOL_APO_PLAN_RUM_MONTH,SEC_VOL_ACTUALS_RUM_MONTH
0,AURG,D3A4,702478,2023-02-28,0.0,0.00
1,AURG,D3A4,702478,2023-03-31,0.0,0.00
2,AURG,D3A4,702478,2023-04-30,0.0,0.00
3,AURG,D3A4,702478,2023-05-31,0.0,0.00
4,AURG,D3A4,702930,2023-02-28,0.0,0.00
...,...,...,...,...,...,...
1177688,None,SNQZ,725845,2025-10-31,0.0,286.56
1177689,None,SNQZ,725845,2025-12-31,0.0,792.00
1177690,None,SNQZ,725845,2026-01-31,0.0,221.04
1177691,None,SNQZ,725845,2026-04-30,0.0,331.20


In [8]:
billwise_query = """ SELECT
    CM.asm_area_code,
    CM.depot_code,
    MM.parent_material_code,
    LAST_DAY(billwise.sales_invoice_date) AS month_date,
    SUM(billwise.sec_vol_actuals_rum) AS sec_actuals_vol_rum
FROM PRD_DB.PUBLIC.DWH_RET_SKU_ACT_DAILY billwise
JOIN (SELECT material_code, parent_material_code, material_group_code FROM PRD_DB.PUBLIC.MST_MATERIAL WHERE company_code='MIL' AND latest_record_ind=1 AND material_code >= '700000') MM ON billwise.material_code = MM.material_code
JOIN (SELECT DISTINCT channel_name, asm_area_code, customer_code, depot_code FROM PRD_DB.PUBLIC.MST_CUSTOMER WHERE company_code='MIL' AND latest_record_ind=1 AND channel_name = '{}') CM ON billwise.distributor_code = CM.customer_code
WHERE billwise.company_code = 'MIL' AND LAST_DAY(billwise.sales_invoice_date) = '{}'
GROUP BY 1, 2, 3, 4 """.format(
    channel, end_date
)



prev_month_df = pd.read_sql(
    con=prod_conn, sql = billwise_query
)
prev_month_df = pd.DataFrame(prev_month_df)
prev_month_df.rename(columns={'SEC_ACTUALS_VOL_RUM':'SEC_VOL_ACTUALS_RUM_MONTH'}, inplace=True)
print('prev_month_df', prev_month_df['SEC_VOL_ACTUALS_RUM_MONTH'].sum())


sales_data = pd.concat(
    [results, prev_month_df],
    ignore_index=True
)
print(f"data is {results.shape} cahnnel is {channel} start_date is {start_date}")
sales_data = sales_data.rename(columns={'parent_material1_code':'parent_material_code'})

prev_month_df 5759425.6767380005
data is (1177693, 6) cahnnel is GT start_date is 2023-01-31


In [15]:
prev_month_df

,ASM_AREA_CODE,DEPOT_CODE,PARENT_MATERIAL_CODE,MONTH_DATE,SEC_VOL_ACTUALS_RUM_MONTH
0,CNI,D674,809586,2026-05-31,-0.000320
1,TPT,D572,808239,2026-05-31,-0.001128
2,RWBN,D232,718309,2026-05-31,0.020000
3,MADU,D674,722329,2026-05-31,-0.018960
4,RAJ2,D314,809057,2026-05-31,-3.600000
...,...,...,...,...,...
10584,CNI,D674,807037,2026-05-31,-0.050000
10585,MUM1,D356,722326,2026-05-31,0.001185
10586,RWBN,D232,725004,2026-05-31,12.656000
10587,AURG,D461,718462,2026-05-31,1.500000


In [10]:
results['MONTH_DATE'].max()

datetime.date(2026, 4, 30)

In [11]:
prev_month_df['MONTH_DATE'].max()

datetime.date(2026, 5, 31)

In [14]:
print(sales_data.groupby(['MONTH_DATE'])['SEC_VOL_ACTUALS_RUM_MONTH'].sum()[20:])

MONTH_DATE
2024-09-30    3.979765e+06
2024-10-31    3.142277e+06
2024-11-30    3.540479e+06
2024-12-31    4.432122e+06
2025-01-31    2.908052e+06
2025-02-28    3.633607e+06
2025-03-31    2.703822e+06
2025-04-30    5.023876e+06
2025-05-31    4.760804e+06
2025-06-30    4.903701e+06
2025-07-31    3.495088e+06
2025-08-31    3.433719e+06
2025-09-30    5.073693e+06
2025-10-31    3.357607e+06
2025-11-30    4.384840e+06
2025-12-31    5.381102e+06
2026-01-31    4.168871e+06
2026-02-28    5.158963e+06
2026-03-31    3.413060e+06
2026-04-30    6.585465e+06
2026-05-31    5.759426e+06
Name: SEC_VOL_ACTUALS_RUM_MONTH, dtype: float64
